## 1 Library Installation, Dataset Loading & Global Config

### Ringkasan Bagian

Bagian ini membangun seluruh fondasi teknis yang dipakai oleh eksperimen kompresi model Donut, sebelum satu pun proses *pruning*, *knowledge distillation* (KD), atau kuantisasi dijalankan. Ada empat hal yang disetup di sini secara berurutan:

1. **Instalasi dependency** —> memasang seluruh library pihak ketiga yang dibutuhkan sepanjang notebook (model, metrik, tracking eksperimen).
2. **Konfigurasi reproducibility** —> mengunci seluruh sumber randomness (Python, NumPy, PyTorch, cuDNN) di bawah satu `SEED` global, supaya hasil antar teknik kompresi bisa dibandingkan secara adil tanpa bias dari variasi acak training.
3. **Konfigurasi experiment tracking** —> menyambungkan notebook ke MLflow yang di-host di Databricks, sehingga setiap run (baseline, pruning, KD, kuantisasi) tercatat otomatis lengkap dengan parameter, metrik, dan artifact model-nya.
4. **Loading dataset dan model baseline** —> mengambil dataset CORD-v2 beserta model Donut pretrained yang menjadi titik awal (baseline) sebelum dikompresi, sekaligus menyesuaikan konfigurasi arsitektur decoder-nya.

---

#### 1.1 Instalasi Dependency

| Library | Fungsi dalam Pipeline |
|---|---|
| `transformers` | Memuat model Donut (`VisionEncoderDecoderModel`) dan `DonutProcessor`, serta `set_seed` untuk reproducibility |
| `datasets` | Memuat dataset CORD-v2 dari HuggingFace Hub |
| `torch` | Backbone training/inference: tensor, autograd, CUDA, *dynamic quantization* |
| `sentencepiece` | Dependency tokenizer yang dipakai `DonutProcessor` |
| `accelerate` | Dependency pendukung loading/inference model HuggingFace |
| `zss` | Menghitung *Tree Edit Distance* untuk metrik N-TED |
| `fvcore` | Estimasi FLOPs model (`FlopCountAnalysis`) |
| `mlflow` (`>=3, <4`) | Experiment tracking: logging parameter, metrik, dan artifact model |
| `databricks-sdk` | Autentikasi dan koneksi ke MLflow tracking server di Databricks |
| `pandas` / `numpy` | Penyusunan tabel hasil eksperimen dan operasi numerik pendukung |

Dan modul standar Python lainnya (`os`, `re`, `json`, `time`, `copy`, `tempfile`, `warnings`, `sys`, `subprocess`, `pathlib`, `random`, `platform`)

#### 1.2 Konfigurasi Reproducibility

Agar perbedaan hasil antar teknik benar-benar mencerminkan efek teknik tersebut dan bukan efek dari randomness training yang berbeda-beda tiap run. Seluruh sumber randomness dikunci ke satu `SEED` melalui fungsi `_set_global_seed`:

| Komponen | Yang Dikunci | Alasan |
|---|---|---|
| `PYTHONHASHSEED`, `random.seed` | Randomness Python murni | Konsistensi operasi berbasis `random` |
| `np.random.seed` | Randomness NumPy | Konsistensi operasi numerik |
| `torch.manual_seed`, `cuda.manual_seed_all` | Randomness inisialisasi tensor & CUDA | Konsistensi bobot/inisialisasi model |
| `transformers.set_seed` | Randomness internal HuggingFace | Konsistensi proses training/generation |
| `cudnn.benchmark=False`, `cudnn.deterministic=True` | Pemilihan algoritma konvolusi cuDNN | Mencegah cuDNN memilih algoritma tercepat yang sifatnya non-deterministik |
| `torch.use_deterministic_algorithms(True, warn_only=True)` | Operasi PyTorch yang punya varian non-deterministik | Memaksa operasi deterministik bila tersedia |
| `CUBLAS_WORKSPACE_CONFIG=":4096:8"` | Determinism operasi cuBLAS (matmul di GPU) | Disyaratkan PyTorch agar `use_deterministic_algorithms` bekerja penuh di CUDA |
| `CPU_NUM_THREADS=1` | Non-determinism dari paralelisme thread CPU | Operasi floating-point paralel bisa menghasilkan urutan penjumlahan berbeda antar run |

Fungsi `reset_experiment_seed` dipanggil ulang sebelum setiap teknik kompresi dijalankan (pruning, KD) di Bagian 8, memastikan setiap konfigurasi rasio pruning mulai dari kondisi acak yang identik.

#### 1.3 Konfigurasi Experiment Tracking (Databricks & MLflow)

Semua hasil eksperimen (metrik, parameter, dan model artifact) dicatat ke MLflow yang di-*host* di Databricks, bukan disimpan manual — sehingga histori eksperimen tetap tertelusuri meskipun runtime Colab berakhir.

| Item | Nilai / Sumber | Fungsi |
|---|---|---|
| `DATABRICKS_HOST`, `DATABRICKS_TOKEN` | Colab Secrets (`google.colab.userdata`) | Kredensial autentikasi; tidak di-*hardcode* di notebook |
| `mlflow.set_tracking_uri("databricks")` | tetap | Mengarahkan MLflow client ke backend Databricks |
| `MLFLOW_EXPERIMENT` | `/Shared/DONUT-CORD-v2-Optimization` | Path experiment tempat seluruh run dikelompokkan |
| `EXPERIMENT_GROUP` | `donut-cord-v2-seed-{SEED}` | Tag pengelompokan run berdasarkan seed, memudahkan filter saat membandingkan run antar seed |

Fungsi `_setup_mlflow_databricks` memverifikasi experiment melalui `MlflowClient.get_experiment` sekali di awal — kalau kredensial atau path experiment salah, notebook gagal secepatnya di sini alih-alih di tengah proses training yang panjang.

#### 1.4 Konfigurasi Global Eksperimen

| Parameter | Nilai | Kategori | Keterangan |
|---|---|---|---|
| `MODEL_NAME` | `naver-clova-ix/donut-base-finetuned-cord-v2` | Model | Model dasar (baseline) yang dikompresi |
| `DATASET_NAME` | `naver-clova-ix/cord-v2` | Dataset | Dataset evaluasi ekstraksi dokumen |
| `MAX_LENGTH` | 512 | Model | Panjang maksimum token target sequence |
| `BATCH_SIZE` | 4 | Training | Dibatasi kapasitas VRAM GPU yang tersedia |
| `PRUNING_RATIOS` | `[0.3, 0.5, 0.7]` | Compression | Rasio *structured pruning* yang diuji (30%/50%/70% neuron FFN decoder dibuang) |
| `KD_EPOCHS` | 5 | Compression | Jumlah epoch fine-tuning distillation setelah pruning |
| `KD_LR` | 5e-5 | Compression | Learning rate AdamW untuk training distillation |
| `KD_TEMPERATURE` | 2.0 | Compression | Softening distribusi logits teacher-student, mengikuti konvensi umum KD (Hinton et al., 2015) |
| `KD_ALPHA` | 0.5 | Compression | Bobot 50% *soft-label loss* (KD) : 50% *hard-label loss* (cross-entropy) |
| `NUM_CALIB_BATCHES` | 25 | Pruning | Jumlah batch kalibrasi untuk menghitung Taylor importance score |
| `EVAL_SAMPLES` | 100 | Evaluation | Jumlah sample test untuk metrik F1 / N-TED |
| `LATENCY_SAMPLES` | 25 | Evaluation | Jumlah sample untuk pengukuran latency inferensi |
| `WARMUP_SAMPLES` | 10 | Evaluation | Jumlah sample warm-up sebelum latency diukur, menghindari bias cold-start GPU |
| `TRAIN_DEVICE` / `EVAL_DEVICE` | `cuda` | Infra | Device eksekusi training dan evaluasi |

#### 1.5 Konfigurasi Model Donut

Donut adalah arsitektur *encoder-decoder*: encoder membaca gambar dokumen, decoder men-*generate* teks terstruktur token demi token. Decoder perlu tahu token khusus mana yang menandai awal, akhir, dan padding sequence, serta ukuran vocabulary-nya — inilah yang diatur `setup_donut_config`:

| Config | Sumber Nilai | Fungsi |
|---|---|---|
| `pad_token_id` | `processor.tokenizer.pad_token_id` | Menandai posisi padding pada target sequence |
| `eos_token_id` | `processor.tokenizer.eos_token_id` | Menandai akhir proses *generate* |
| `decoder_start_token_id` | Token `<s_cord-v2>` | Prompt awal decoder — token task-specific untuk skema CORD-v2 |
| `vocab_size` | `model.config.decoder.vocab_size` | Menyelaraskan ukuran output layer decoder dengan ukuran vocabulary tokenizer |

Konfigurasi ini diterapkan ke `baseline_model`, dan diterapkan ulang setiap kali model hasil pruning dibentuk (`build_pruned_model` di Bagian 4), karena `deepcopy` model tidak otomatis mewarisi konfigurasi khusus ini.

#### 1.6 Loading Dataset & Model Baseline

Dataset CORD-v2 dimuat langsung dari HuggingFace Hub menggunakan split bawaan (`train` / `validation` / `test`) — tanpa split manual — supaya pembagian data konsisten dengan benchmark yang biasa dipakai pada literatur terkait. `DonutProcessor` (image processor + tokenizer) dan `baseline_model` dimuat dari checkpoint pretrained yang sama (`MODEL_NAME`), menjadi titik acuan (baseline) yang dibandingkan dengan seluruh model hasil kompresi pada Bagian 9.


In [ ]:
!pip install -q "mlflow>=3, <4" databricks-sdk transformers datasets sentencepiece accelerate zss fvcore

In [ ]:
import os
import re
import json
import time
import copy
import tempfile
import warnings

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from PIL import Image
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import DonutProcessor, VisionEncoderDecoderModel

from zss import Node, simple_distance
from fvcore.nn import FlopCountAnalysis

warnings.filterwarnings("ignore")

In [ ]:
import os
import sys
import random
import platform
import subprocess
from pathlib import Path

import numpy as np
import mlflow
import transformers

from google.colab import userdata
from mlflow.tracking import MlflowClient

In [ ]:
# Reproducibility Config
SEED = 27
CPU_NUM_THREADS = 1
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
def _set_global_seed(seed=SEED):
    """ """
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    transformers.set_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

    torch.use_deterministic_algorithms(True, warn_only=True)


_set_global_seed(SEED)
torch.set_num_threads(CPU_NUM_THREADS)

In [ ]:
# MLFlow DataBricks Config
MLFLOW_EXPERIMENT = "/Shared/DONUT-CORD-v2-Optimization"
EXPERIMENT_GROUP = f"donut-cord-v2-seed-{SEED}"


def _setup_mlflow_databricks():
    """ """

    try:
        databricks_host = userdata.get("DATABRICKS_HOST")
        databricks_token = userdata.get("DATABRICKS_TOKEN")
    except Exception as e:
        raise RuntimeError(
            "Secrets belum di setup untuk DATABRICKS_HOST dan DATABRICKS_TOKEN"
        ) from e

    databricks_host = databricks_host.strip().rstrip("/")

    if not databricks_host.startswith(("http://", "https://")):
        databricks_host = "https://" + databricks_host

    os.environ["DATABRICKS_HOST"] = databricks_host
    os.environ["DATABRICKS_TOKEN"] = databricks_token

    mlflow.set_tracking_uri("databricks")

    experiment = mlflow.set_experiment(MLFLOW_EXPERIMENT)

    client = MlflowClient()
    client.get_experiment(experiment.experiment_id)

    return experiment


MLFLOW_EXPERIMENT_INFO = _setup_mlflow_databricks()

In [ ]:
MODEL_NAME = "naver-clova-ix/donut-base-finetuned-cord-v2"
DATASET_NAME = "naver-clova-ix/cord-v2"

MAX_LENGTH = 512
BATCH_SIZE = 4

PRUNING_RATIOS = [0.3, 0.5, 0.7]

KD_EPOCHS = 20
KD_LR = 5e-5
KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5  # Weight: 50% KD loss + 50% Cross Entropy loss

NUM_CALIB_BATCHES = 25  # Num Batch untuk Gradient Calibration
EVAL_SAMPLES = 100
LATENCY_SAMPLES = 25
WARMUP_SAMPLES = 10

TRAIN_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EVAL_DEVICE = "cuda"

print("Train device:", TRAIN_DEVICE)
print("Eval device :", EVAL_DEVICE)

In [ ]:
dataset = load_dataset(DATASET_NAME)

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

processor = DonutProcessor.from_pretrained(MODEL_NAME)
baseline_model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)

In [ ]:
def setup_donut_config(model, processor):
    model.config.pad_token_id = processor.tokenizer.pad_token_id
    model.config.eos_token_id = processor.tokenizer.eos_token_id
    model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(
        "<s_cord-v2>"
    )
    model.config.vocab_size = model.config.decoder.vocab_size
    return model


baseline_model = setup_donut_config(baseline_model, processor)
baseline_model.to(TRAIN_DEVICE)

print("Train data:", len(train_data))
print("Validation data:", len(val_data))
print("Test data:", len(test_data))

## 2 Preprocessing & Data Pipeline

In [ ]:
def json2token(obj):
    """
    Mengubah objek JSON ground truth menjadi sequence token tag Donut.

    Donut tidak memprediksi JSON secara langsung, melainkan string linear
    bertag XML-like: setiap key `k` menjadi pasangan `<s_k> ... </s_k>`, dan
    antar-elemen list dipisah token `<sep/>`. Fungsi ini adalah kebalikan dari
    `processor.token2json` yang dipakai saat inferensi.

    Contoh:
        {"total": 12} -> "<s_total>12</s_total>"

    Args:
        obj: dict, list, atau nilai skalar dari `gt_parse` CORD-v2.

    Returns:
        str sequence token target.
    """
    if isinstance(obj, dict):
        sequence = ""
        for key, value in obj.items():
            sequence += f"<s_{key}>"
            sequence += json2token(value)
            sequence += f"</s_{key}>"
        return sequence

    elif isinstance(obj, list):
        return "<sep/>".join(json2token(item) for item in obj)

    else:
        return str(obj)

In [ ]:
class CordDataset(Dataset):
    """
    Dataset PyTorch yang mengubah sample CORD-v2 menjadi pasangan (gambar, label token).

    Satu sample menghasilkan `pixel_values` hasil preprocessing gambar oleh
    DonutProcessor, dan `labels` berupa id token dari `gt_parse` yang sudah
    dilinearisasi `json2token`.
    """

    def __init__(self, data, processor, max_length=512):
        """
        Args:
            data: split HuggingFace dataset CORD-v2 (train/validation/test).
            processor: DonutProcessor (image processor + tokenizer).
            max_length: panjang maksimum target sequence; kelebihannya dipotong.
        """
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        """Jumlah sample dalam split."""
        return len(self.data)

    def __getitem__(self, idx):
        """
        Menyiapkan satu sample siap-training.

        Alur: gambar dikonversi ke RGB lalu diproses menjadi `pixel_values`;
        ground truth JSON dilinearisasi menjadi `<s_cord-v2>...<eos>` lalu
        di-tokenize dengan padding ke `max_length`.

        Token padding di-set ke -100 supaya diabaikan `F.cross_entropy`
        (`ignore_index=-100`). Tanpa ini loss ikut menghitung posisi kosong dan
        model belajar memprediksi padding alih-alih isi dokumen.

        Args:
            idx: indeks sample.

        Returns:
            dict berisi `pixel_values` (tensor gambar) dan `labels` (tensor id token).
        """
        sample = self.data[idx]

        image = sample["image"].convert("RGB")

        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(
            0
        )

        ground_truth = json.loads(sample["ground_truth"])
        gt_parse = ground_truth["gt_parse"]

        target_sequence = (
            "<s_cord-v2>" + json2token(gt_parse) + self.processor.tokenizer.eos_token
        )

        labels = self.processor.tokenizer(
            target_sequence,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)

        labels[labels == self.processor.tokenizer.pad_token_id] = -100

        return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
def seed_worker(workers):
    """
    Menyeragamkan seed tiap worker DataLoader agar pengambilan data reproducible.

    DataLoader dengan `num_workers > 0` menjalankan proses terpisah yang
    masing-masing punya state random sendiri. Tanpa `worker_init_fn`, urutan
    data yang dihasilkan bisa berbeda antar run meskipun seed global sudah
    dikunci di `_set_global_seed`.

    Catatan: ekspresi `% (2 * 32)` memampatkan seed ke rentang 0..63, sehingga
    worker yang berbeda berpeluang besar mendapat seed identik. Nilai yang lazim
    dipakai adalah `2 ** 32`. Perilaku ini dibiarkan apa adanya agar konsisten
    dengan run yang sudah tercatat di MLflow.

    Args:
        workers: indeks worker dari PyTorch. Tidak dipakai; seed diambil dari
            `torch.initial_seed()` yang sudah unik per worker.
    """
    worker_seed = torch.initial_seed() % (2 * 32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


train_generator = torch.Generator()
train_generator.manual_seed(SEED)

In [ ]:
train_dataset = CordDataset(train_data, processor, MAX_LENGTH)
val_dataset = CordDataset(val_data, processor, MAX_LENGTH)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    worker_init_fn=seed_worker,
    generator=train_generator,
)

calibration_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    worker_init_fn=seed_worker,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    worker_init_fn=seed_worker,
)

In [ ]:
def reset_experiment_seed(seed=SEED):
    """
    Mengembalikan seluruh sumber randomness ke kondisi awal sebelum satu teknik dijalankan.

    Dipanggil sebelum setiap konfigurasi di Bagian 8 (pruning, KD), sehingga
    perbedaan hasil antar rasio benar-benar berasal dari teknik kompresinya dan
    bukan dari state random sisa eksperimen sebelumnya.

    Selain seed global, generator DataLoader (`train_generator`) ikut di-reset
    supaya urutan shuffling batch training identik di tiap run.

    Args:
        seed: nilai seed; default `SEED` global.
    """
    _set_global_seed(seed)
    train_generator.manual_seed(seed)

## 3 Evaluation Metrics & Utilities

In [ ]:
def clean_sequence(sequence):
    """
    Membersihkan token khusus dari hasil decode sebelum diparsing menjadi JSON.

    `batch_decode` sengaja dipanggil dengan `skip_special_tokens=False` supaya
    tag struktur `<s_key>`/`</s_key>` tetap ada. Konsekuensinya token EOS dan
    PAD ikut terbawa dan harus dibuang manual di sini.

    Args:
        sequence: string hasil `processor.batch_decode`.

    Returns:
        str tanpa token EOS/PAD dan tanpa spasi di ujung.
    """
    sequence = sequence.replace(processor.tokenizer.eos_token, "")
    sequence = sequence.replace(processor.tokenizer.pad_token, "")
    sequence = sequence.strip()
    return sequence

In [ ]:
def predict_json(model, image, device="cpu"):
    """
    Menjalankan inferensi satu gambar dokumen dan mengembalikan hasilnya sebagai JSON.

    Decoding dibuat deterministik (`do_sample=False`, `num_beams=1` alias greedy)
    supaya perbandingan antar model tidak terkontaminasi variasi sampling: model
    yang sama selalu menghasilkan output yang sama untuk gambar yang sama.

    Bila sequence yang dihasilkan rusak secara struktur (tag tidak berpasangan),
    `token2json` gagal dan fungsi mengembalikan dict kosong. Prediksi tersebut
    diperlakukan sebagai jawaban yang seluruh field-nya salah, bukan sebagai
    error yang menghentikan proses evaluasi.

    Args:
        model: model Donut yang dievaluasi.
        image: PIL Image dokumen.
        device: device inferensi ("cuda"/"cpu"). Model terkuantisasi wajib "cpu".

    Returns:
        tuple `(pred_json, sequence)` berisi hasil parsing dan string mentahnya.
    """
    model.eval()
    model.to(device)

    image = image.convert("RGB")

    pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)

    decoder_input_ids = processor.tokenizer(
        "<s_cord-v2>", add_special_tokens=False, return_tensors="pt"
    ).input_ids.to(device)

    with torch.no_grad():
        output_ids = model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            max_length=MAX_LENGTH,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            # deterministic decoding
            do_sample=False,
            num_beams=1,
        )

    sequence = processor.batch_decode(output_ids, skip_special_tokens=False)[0]
    sequence = clean_sequence(sequence)

    try:
        pred_json = processor.token2json(sequence)
    except:
        pred_json = {}

    return pred_json, sequence

In [ ]:
def normalize_text(text):
    """
    Menormalkan nilai field sebelum dibandingkan dengan ground truth.

    Lowercase dan perapian whitespace supaya perbedaan kapitalisasi atau spasi
    ganda tidak dihitung sebagai kesalahan ekstraksi.

    Args:
        text: nilai apa pun; dikonversi ke str lebih dulu.

    Returns:
        str hasil normalisasi.
    """
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

In [ ]:
def flatten_json(obj, prefix=""):
    """
    Meratakan JSON bersarang menjadi dict datar berkunci path.

    Perbandingan field-level membutuhkan unit yang bisa dicocokkan satu per satu.
    Struktur bersarang diratakan menjadi bentuk seperti `menu.0.nm` -> "es teh",
    sehingga kesamaan prediksi dan ground truth dapat dihitung sebagai irisan
    himpunan pasangan (path, nilai).

    Args:
        obj: dict, list, atau skalar yang akan diratakan.
        prefix: path induk; dipakai internal saat rekursi.

    Returns:
        dict {path: nilai ternormalisasi}.
    """
    items = {}

    if isinstance(obj, dict):
        for key, value in obj.items():
            new_key = f"{prefix}.{key}" if prefix else key
            items.update(flatten_json(value, new_key))

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            new_key = f"{prefix}.{i}"
            items.update(flatten_json(value, new_key))

    else:
        items[prefix] = normalize_text(obj)

    return items

In [ ]:
def field_level_f1(pred_json, true_json):
    """
    Menghitung Precision, Recall, dan F1 pada level pasangan (field, nilai).

    Prediksi dan ground truth diratakan lalu diperlakukan sebagai himpunan:
    true positive adalah pasangan yang path sekaligus nilainya sama persis.
    Metrik ini bersifat exact-match per field, sehingga nilai yang benar tetapi
    ditempatkan pada path yang salah dihitung sebagai false positive sekaligus
    false negative.

    Konstanta 1e-8 pada penyebut mencegah pembagian nol ketika model gagal
    menghasilkan JSON dan prediksinya kosong.

    Args:
        pred_json: hasil ekstraksi model.
        true_json: `gt_parse` ground truth.

    Returns:
        tuple `(precision, recall, f1)`.
    """
    pred_items = set(flatten_json(pred_json).items())
    true_items = set(flatten_json(true_json).items())

    tp = len(pred_items & true_items)
    fp = len(pred_items - true_items)
    fn = len(true_items - pred_items)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1

In [ ]:
def json_to_tree(obj, name="root"):
    """
    Mengubah JSON menjadi pohon `zss.Node` untuk perhitungan Tree Edit Distance.

    F1 field-level tidak melihat struktur: prediksi dengan seluruh field benar
    tetapi hierarki salah tetap memperoleh nilai tinggi. Representasi pohon
    dipakai agar kesalahan struktur ikut terukur — key menjadi node internal,
    nilai menjadi node daun, dan elemen list diberi label posisi `item_i`.

    Args:
        obj: dict, list, atau skalar.
        name: label node saat ini.

    Returns:
        `zss.Node` akar dari subtree.
    """
    node = Node(str(name))

    if isinstance(obj, dict):
        for key, value in obj.items():
            child = json_to_tree(value, key)
            node.addkid(child)

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            child = json_to_tree(value, f"item_{i}")
            node.addkid(child)

    else:
        value_node = Node(normalize_text(obj))
        node.addkid(value_node)

    return node

In [ ]:
def count_nodes(node):
    """
    Menghitung total node dalam sebuah pohon secara rekursif.

    Dipakai sebagai normalizer N-TED agar jarak edit tidak bias terhadap ukuran
    dokumen: struk yang panjang otomatis punya jarak edit lebih besar meskipun
    kualitas ekstraksinya setara.

    Args:
        node: `zss.Node` akar.

    Returns:
        int jumlah node termasuk akar.
    """
    total = 1
    for child in node.children:
        total += count_nodes(child)
    return total

In [ ]:
def normalized_tree_edit_distance(pred_json, true_json):
    """
    Menghitung Normalized Tree Edit Distance (N-TED) antara prediksi dan ground truth.

    TED adalah jumlah minimum operasi (insert, delete, rename node) yang
    dibutuhkan untuk mengubah pohon prediksi menjadi pohon ground truth;
    `label_distance` memberi biaya 1 untuk label berbeda dan 0 untuk label sama.
    Hasilnya dibagi jumlah node pohon terbesar supaya sebanding antar dokumen
    dengan ukuran berbeda.

    Arah metrik ini berlawanan dengan F1: **semakin kecil semakin baik**, 0
    berarti struktur prediksi identik dengan ground truth.

    Args:
        pred_json: hasil ekstraksi model.
        true_json: `gt_parse` ground truth.

    Returns:
        float N-TED.
    """
    pred_tree = json_to_tree(pred_json)
    true_tree = json_to_tree(true_json)

    def label_distance(a, b):
        return 0 if a == b else 1

    distance = simple_distance(pred_tree, true_tree, label_dist=label_distance)

    normalizer = max(count_nodes(pred_tree), count_nodes(true_tree), 1)
    nted = distance / normalizer

    return nted

In [ ]:
def evaluate_extraction(model, data, device="cpu", max_samples=100):
    """
    Mengukur kualitas ekstraksi model pada sejumlah sample test.

    Prediksi dijalankan satu per satu (batch size 1) karena `generate`
    menghasilkan sequence dengan panjang berbeda-beda tiap dokumen. Skor tiap
    sample dirata-rata secara makro: setiap dokumen berbobot sama, tidak peduli
    berapa banyak field yang dikandungnya.

    Args:
        model: model yang dievaluasi.
        data: split dataset, umumnya `test_data`.
        device: device inferensi.
        max_samples: batas jumlah sample. Dibatasi demi waktu evaluasi, karena
            10 konfigurasi model harus dievaluasi berurutan.

    Returns:
        dict berisi rata-rata `Precision`, `Recall`, `F1`, dan `N-TED`.
    """
    precision_scores = []
    recall_scores = []
    f1_scores = []
    nted_scores = []

    total_samples = min(len(data), max_samples)

    for i in range(total_samples):
        sample = data[i]

        true_json = json.loads(sample["ground_truth"])["gt_parse"]

        pred_json, _ = predict_json(model, sample["image"], device=device)

        precision, recall, f1 = field_level_f1(pred_json, true_json)

        nted = normalized_tree_edit_distance(pred_json, true_json)

        precision_scores.append(precision)
        recall_scores.append(recall)
        f1_scores.append(f1)
        nted_scores.append(nted)

    return {
        "Precision": sum(precision_scores) / len(precision_scores),
        "Recall": sum(recall_scores) / len(recall_scores),
        "F1": sum(f1_scores) / len(f1_scores),
        "N-TED": sum(nted_scores) / len(nted_scores),
    }

In [ ]:
def get_model_size_mb(model):
    """
    Mengukur ukuran model dari besar file state_dict yang diserialisasi.

    Menghitung `numel * itemsize` tidak valid untuk model terkuantisasi, karena
    tensor INT8 menyimpan skala dan zero-point tambahan. Menyimpan state_dict ke
    file sementara memberi angka yang apple-to-apple antara model FP32 dan INT8,
    yaitu ukuran nyata yang harus di-deploy.

    File sementara selalu dihapus di blok `finally`, termasuk bila penyimpanan
    gagal di tengah jalan.

    Args:
        model: model yang diukur.

    Returns:
        float ukuran dalam MB.
    """
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pt") as tmp:

        temporary_path = tmp.name

    try:
        torch.save(model.state_dict(), temporary_path)

        size_mb = os.path.getsize(temporary_path) / (1024 * 1024)

    finally:
        if os.path.exists(temporary_path):
            os.remove(temporary_path)

    return size_mb

In [ ]:
def measure_latency(model, data, device="cpu", num_samples=100):
    """
    Mengukur rata-rata waktu inferensi per dokumen dalam milidetik.

    Tahap warm-up sebanyak `WARMUP_SAMPLES` dijalankan lebih dulu dan hasilnya
    dibuang, karena eksekusi pertama menanggung biaya sekali-jalan berupa
    alokasi memori CUDA, kompilasi kernel, dan pengisian cache — biaya yang
    tidak mencerminkan latensi kondisi steady-state.

    Yang diukur adalah seluruh proses generate autoregresif sampai token EOS,
    bukan satu forward pass, sehingga angka inilah yang relevan untuk deployment.

    Perbandingan latensi hanya valid bila `device` sama antar model. Model
    terkuantisasi wajib diukur di CPU, dan pembandingnya pun harus di CPU
    (lihat catatan metodologis Bagian 8).

    Args:
        model: model yang diukur.
        data: split dataset sumber gambar.
        device: device pengukuran.
        num_samples: jumlah sample yang diukur setelah warm-up.

    Returns:
        float rata-rata latensi dalam milidetik per sample.
    """
    model.eval()
    model.to(device)

    # Warm-Up
    warmup_samples = min(WARMUP_SAMPLES, len(data))

    for i in range(warmup_samples):
        image = data[i]["image"].convert("RGB")

        pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
        decoder_input_ids = processor.tokenizer(
            "<s_cord-v2>", add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(device)

        with torch.no_grad():
            _ = model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=MAX_LENGTH,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                use_cache=True,
            )

    # Latency measurement
    times = []
    total_samples = min(len(data), num_samples)

    for i in range(total_samples):
        image = data[i]["image"].convert("RGB")

        pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
        decoder_input_ids = processor.tokenizer(
            "<s_cord-v2>", add_special_tokens=False, return_tensors="pt"
        ).input_ids.to(device)

        start = time.time()

        with torch.no_grad():
            _ = model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                max_length=MAX_LENGTH,
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
                use_cache=True,
            )

        end = time.time()
        times.append(end - start)

    return sum(times) / len(times) * 1000

In [ ]:
class DonutFlopsWrapper(nn.Module):
    """
    Pembungkus model Donut agar dapat ditelusuri `fvcore.FlopCountAnalysis`.

    `FlopCountAnalysis` menelusuri graf komputasi lewat satu pemanggilan forward
    dengan argumen positional dan mengharapkan keluaran berupa tensor. Sementara
    itu VisionEncoderDecoderModel menerima argumen keyword dan mengembalikan
    objek output HuggingFace. Wrapper ini menyederhanakan signature-nya menjadi
    `(pixel_values, decoder_input_ids) -> logits`.

    `use_cache=False` dipakai supaya yang terhitung adalah satu forward pass
    penuh (teacher forcing), bukan langkah decoding yang memakai cache — bentuk
    inilah yang sebanding antar model.
    """

    def __init__(self, model):
        """
        Args:
            model: VisionEncoderDecoderModel yang akan dihitung FLOPs-nya.
        """
        super().__init__()
        self.model = model

    def forward(self, pixel_values, decoder_input_ids):
        """
        Args:
            pixel_values: tensor gambar hasil DonutProcessor.
            decoder_input_ids: tensor id token input decoder.

        Returns:
            Tensor logits decoder.
        """
        outputs = self.model(
            pixel_values=pixel_values,
            decoder_input_ids=decoder_input_ids,
            use_cache=False,
        )
        return outputs.logits

In [ ]:
def estimate_flops_gflops(model):
    """
    Mengestimasi beban komputasi model dalam GFLOPs untuk satu dokumen.

    Berbeda dengan latensi yang bergantung pada kondisi hardware, FLOPs adalah
    ukuran kompleksitas yang independen device. Metrik ini dipakai untuk
    membuktikan bahwa pruning benar-benar memangkas komputasi, bukan sekadar
    tampak lebih cepat karena kondisi mesin saat pengukuran.

    Perhitungan dilakukan pada salinan model di CPU agar tidak mengganggu state
    maupun alokasi memori GPU model asli. `decoder_input_ids` diisi penuh
    sepanjang `MAX_LENGTH` supaya semua model diukur pada panjang sequence yang
    sama.

    `FlopCountAnalysis` gagal pada modul yang tidak dikenalinya, misalnya layer
    terkuantisasi. Kegagalan ditangkap dan fungsi mengembalikan None sehingga
    proses evaluasi tetap berjalan; untuk model terkuantisasi FLOPs diambil dari
    versi FP32-nya lewat parameter `flops_source_model` pada `evaluate_model`.

    Args:
        model: model yang dihitung.

    Returns:
        float GFLOPs, atau None bila perhitungan gagal.
    """
    model_copy = copy.deepcopy(model)
    model_copy.eval()
    model_copy.to("cpu")

    sample = test_data[0]
    image = sample["image"].convert("RGB")

    pixel_values = processor(image, return_tensors="pt").pixel_values

    decoder_input_ids = torch.full(
        size=(1, MAX_LENGTH),
        fill_value=processor.tokenizer.pad_token_id,
        dtype=torch.long,
    )

    decoder_input_ids[:, 0] = model_copy.config.decoder_start_token_id

    wrapper = DonutFlopsWrapper(model_copy)

    try:
        flops = FlopCountAnalysis(wrapper, (pixel_values, decoder_input_ids))
        flops.unsupported_ops_warnings(False)
        flops.uncalled_modules_warnings(False)
        return flops.total() / 1e9

    except Exception as e:
        print("FLOPs gagal dihitung:", e)
        return None

In [ ]:
def evaluate_model(model_name, model, flops_source_model=None, device=None):
    """
    Menjalankan seluruh pengukuran untuk satu konfigurasi model.

    Titik masuk tunggal evaluasi: menggabungkan metrik kualitas (Precision,
    Recall, F1, N-TED) dan metrik efisiensi (ukuran, latensi, FLOPs) menjadi
    satu baris hasil dengan kunci yang konsisten, sehingga seluruh konfigurasi
    dapat disusun menjadi satu tabel perbandingan di Bagian 9.

    Args:
        model_name: label model pada tabel hasil dan MLflow, mis. "DONUT-P50-KD".
        model: model yang dievaluasi.
        flops_source_model: model alternatif sebagai sumber perhitungan FLOPs.
            Dipakai untuk model terkuantisasi yang tidak dapat ditelusuri fvcore,
            diisi dengan versi FP32-nya.
        device: device evaluasi; default `EVAL_DEVICE`. Model terkuantisasi wajib
            diberi "cpu" karena kernel INT8 dinamis tidak tersedia di CUDA.

    Returns:
        dict satu baris hasil evaluasi.
    """
    if device is None:
        device = EVAL_DEVICE

    print(f"\nEvaluasi: {model_name} (device: {device})")

    extraction_metrics = evaluate_extraction(
        model, test_data, device=device, max_samples=EVAL_SAMPLES
    )

    latency = measure_latency(
        model, test_data, device=device, num_samples=LATENCY_SAMPLES
    )

    size_mb = get_model_size_mb(model)

    if flops_source_model is None:
        flops_source_model = model

    flops = estimate_flops_gflops(flops_source_model)

    result = {
        "Model": model_name,
        "Precision": extraction_metrics["Precision"],
        "Recall": extraction_metrics["Recall"],
        "F1-Score": extraction_metrics["F1"],
        "N-TED": extraction_metrics["N-TED"],
        "Size (MB)": size_mb,
        "Latency (ms/sample)": latency,
        "FLOPs (GFLOPs)": flops,
    }

    return result

## 4 Structured Pruning (Taylor Importance)

In [ ]:
def get_decoder_layers(model):
    """
    Mengambil daftar layer decoder Donut dari struktur model yang bersarang.

    Path-nya panjang karena Donut adalah `VisionEncoderDecoderModel`: `model.decoder`
    masih berupa `MBartForCausalLM`, di dalamnya ada wrapper `.model`, baru kemudian
    `.decoder.layers` yang berisi `MBartDecoderLayer` sesungguhnya. Dibungkus jadi
    satu fungsi supaya path ini hanya ditulis di satu tempat — seluruh tahap pruning
    (skoring, pemotongan, verifikasi) menunjuk ke objek layer yang sama.

    Args:
        model: model Donut (`VisionEncoderDecoderModel`).

    Returns:
        `nn.ModuleList` berisi seluruh layer decoder.
    """
    return model.decoder.model.decoder.layers


In [ ]:
def collect_taylor_scores(model, dataloader, num_batches=5, device="cuda"):
    """
    Menghitung skor pentingnya tiap neuron FFN decoder dengan kriteria Taylor.

    Skor Taylor orde satu memperkirakan seberapa besar loss akan berubah bila sebuah
    neuron dinolkan, yaitu `|w * dL/dw|`. Bobot yang besar tetapi tidak pernah
    memengaruhi loss (gradien kecil) karena itu tetap dianggap tidak penting —
    berbeda dengan kriteria magnitude yang hanya melihat besar bobot.

    Satu neuron FFN hidup di dua matriks sekaligus: sebagai *baris* pada `fc1` dan
    sebagai *kolom* pada `fc2`. Kontribusi keduanya dijumlahkan (`dim=1` untuk fc1,
    `dim=0` untuk fc2) agar satu neuron menghasilkan satu skor.

    Gradien diakumulasikan dari beberapa batch, bukan satu batch, supaya skor tidak
    bergantung pada segelintir dokumen. Loss dibagi `num_batches` sebelum `backward()`
    sehingga hasil akumulasinya setara rata-rata, bukan jumlah. `use_cache=False`
    dipakai karena KV cache tidak diperlukan saat forward untuk mencari gradien dan
    hanya memakan memori.

    Skor dipindahkan ke CPU dan di-`detach` agar graf komputasi maupun VRAM tidak
    tertahan setelah fungsi selesai; `zero_grad()` di akhir memastikan model yang
    dioper ke tahap berikutnya bersih dari gradien sisa.

    Args:
        model: model baseline yang menjadi acuan penilaian.
        dataloader: sumber batch, umumnya `train_dataloader`.
        num_batches: jumlah batch yang diakumulasikan untuk estimasi gradien.
        device: device tempat forward/backward dijalankan.

    Returns:
        dict `{layer_idx: tensor skor per neuron}` pada CPU, panjang tiap tensor
        sama dengan `decoder_ffn_dim` sebelum pruning.
    """
    model.eval()
    model.to(device)
    model.zero_grad()

    for param in model.parameters():
        param.grad = None

    for batch_idx, batch in enumerate(dataloader):
        if batch_idx >= num_batches:
            break

        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels, use_cache=False)

        loss = outputs.loss / num_batches
        loss.backward()

    scores = {}
    layers = get_decoder_layers(model)

    for layer_idx, layer in enumerate(layers):
        fc1 = layer.fc1
        fc2 = layer.fc2

        score_fc1 = torch.abs(fc1.weight * fc1.weight.grad).sum(dim=1)
        score_fc2 = torch.abs(fc2.weight * fc2.weight.grad).sum(dim=0)

        score = score_fc1 + score_fc2

        if fc1.bias is not None and fc1.bias.grad is not None:
            score += torch.abs(fc1.bias * fc1.bias.grad)

        scores[layer_idx] = score.detach().cpu()

    model.zero_grad()
    return scores


In [ ]:
def prune_decoder_ffn_layer(layer, keep_indices):
    """
    Mengganti `fc1`/`fc2` sebuah layer decoder dengan versi yang lebih sempit.

    Inilah yang membuat pruning ini bersifat *structured*: neuron yang dibuang benar-benar
    hilang dari model karena `nn.Linear` dibangun ulang dengan dimensi baru, bukan
    sekadar dinolkan lewat mask. Hanya dengan cara ini jumlah parameter, ukuran file,
    dan FLOPs ikut turun — masking menyisakan bobot nol yang tetap disimpan dan tetap
    dikalikan.

    Kedua matriks dipotong pada sumbu yang berbeda namun dengan indeks yang sama:
    `fc1` dipotong pada dimensi output (baris) dan `fc2` pada dimensi input (kolom),
    karena keduanya merujuk neuron tersembunyi yang identik. Bias `fc2` sengaja tidak
    disentuh sebab ukurannya mengikuti dimensi output layer, yang tidak berubah.

    `.clone()` dipakai agar bobot baru tidak menyimpan view ke tensor lama, sehingga
    memori matriks asli benar-benar dapat dibebaskan. Layer dimodifikasi *in place*.

    Args:
        layer: satu `MBartDecoderLayer` yang akan dipangkas.
        keep_indices: indeks neuron FFN yang dipertahankan.

    Returns:
        None — perubahan dilakukan langsung pada `layer`.
    """
    fc1 = layer.fc1
    fc2 = layer.fc2

    device = fc1.weight.device
    dtype = fc1.weight.dtype

    keep_indices = keep_indices.to(device)

    new_fc1 = nn.Linear(
        in_features=fc1.in_features,
        out_features=len(keep_indices),
        bias=fc1.bias is not None,
    ).to(device=device, dtype=dtype)

    new_fc2 = nn.Linear(
        in_features=len(keep_indices),
        out_features=fc2.out_features,
        bias=fc2.bias is not None,
    ).to(device=device, dtype=dtype)

    new_fc1.weight.data = fc1.weight.data[keep_indices, :].clone()

    if fc1.bias is not None:
        new_fc1.bias.data = fc1.bias.data[keep_indices].clone()

    new_fc2.weight.data = fc2.weight.data[:, keep_indices].clone()

    if fc2.bias is not None:
        new_fc2.bias.data = fc2.bias.data.clone()

    layer.fc1 = new_fc1
    layer.fc2 = new_fc2


In [ ]:
def apply_taylor_pruning(model, taylor_scores, pruning_ratio):
    """
    Memangkas seluruh FFN decoder sesuai rasio pruning yang diminta.

    Rasio diterapkan secara seragam per layer: setiap layer decoder membuang
    proporsi neuron yang sama, bukan bersaing dalam satu peringkat global. Pilihan ini
    menjaga dimensi FFN tetap sama di semua layer sehingga arsitektur hasil pruning
    masih dapat dideskripsikan oleh satu nilai `decoder_ffn_dim` — syarat agar model
    bisa disimpan dan dimuat ulang lewat `from_pretrained` seperti biasa.

    Yang dipangkas hanya FFN pada decoder; encoder Swin dibiarkan utuh. Konsekuensinya
    penghematan ukuran dan FLOPs terbatas, karena sebagian besar komputasi Donut ada
    di encoder visual.

    `torch.topk` memilih neuron berskor tertinggi, lalu indeksnya diurutkan kembali
    supaya urutan neuron asli tetap terjaga dan hasil pruning deterministik.
    `max(1, ...)` menjaga agar layer tidak pernah kosong sepenuhnya pada rasio ekstrem.

    Pembaruan `model.config.decoder.decoder_ffn_dim` di akhir bersifat wajib: tanpa itu
    config masih mencatat dimensi lama dan model hasil pruning akan gagal dimuat ulang.

    Args:
        model: model yang akan dipangkas (dimodifikasi in place).
        taylor_scores: hasil `collect_taylor_scores` dari model baseline.
        pruning_ratio: proporsi neuron yang dibuang, mis. `0.5` untuk 50%.

    Returns:
        model yang sama setelah dipangkas, dengan config yang sudah disesuaikan.
    """
    layers = get_decoder_layers(model)
    new_ffn_dim = None

    for layer_idx, layer in enumerate(layers):
        scores = taylor_scores[layer_idx]

        total_neurons = len(scores)
        keep_neurons = int(total_neurons * (1 - pruning_ratio))
        keep_neurons = max(1, keep_neurons)

        keep_indices = torch.topk(scores, keep_neurons).indices
        keep_indices = torch.sort(keep_indices).values

        prune_decoder_ffn_layer(layer, keep_indices)
        new_ffn_dim = len(keep_indices)

    model.config.decoder.decoder_ffn_dim = new_ffn_dim
    return model


## 5 Knowledge Distillation

In [ ]:
def build_pruned_model(baseline_model, taylor_scores, pruning_ratio):
    """
    Menyiapkan student model: salinan baseline yang sudah dipangkas dan siap dilatih.

    `copy.deepcopy` wajib di sini karena `apply_taylor_pruning` memodifikasi model
    secara in place. Tanpa penyalinan, baseline akan ikut terpangkas dan tidak lagi
    bisa dipakai sebagai teacher — sekaligus merusak perbandingan terhadap baseline
    pada seluruh eksperimen berikutnya.

    Urutannya penting: model dipindahkan ke `TRAIN_DEVICE` *sebelum* dipangkas, sebab
    `prune_decoder_ffn_layer` membaca device dari bobot lama untuk menentukan tempat
    `nn.Linear` pengganti dibuat. Jika dibalik, layer baru akan lahir di CPU sementara
    sisa model ada di GPU.

    `setup_donut_config` dipanggil ulang agar salinan membawa `pad_token_id`,
    `eos_token_id`, dan `decoder_start_token_id` yang benar, bukan mewarisi config
    mentah dari checkpoint.

    Args:
        baseline_model: model acuan yang tidak boleh ikut berubah.
        taylor_scores: skor pentingnya neuron dari `collect_taylor_scores`.
        pruning_ratio: proporsi neuron FFN yang dibuang, mis. `0.5`.

    Returns:
        student model hasil pruning, sudah berada di `TRAIN_DEVICE`.
    """
    student_model = copy.deepcopy(baseline_model)
    student_model = setup_donut_config(student_model, processor)
    student_model.to(TRAIN_DEVICE)

    student_model = apply_taylor_pruning(student_model, taylor_scores, pruning_ratio)

    print(f"Pruning {int(pruning_ratio*100)}%")
    return student_model


In [ ]:
def count_parameters(model):
    """
    Menghitung total parameter model.

    Dipakai untuk memverifikasi bahwa pruning benar-benar mengurangi parameter, bukan
    sekadar menonaktifkannya. Seluruh parameter dihitung, termasuk yang beku
    (`requires_grad=False`), karena yang diukur adalah ukuran model — bukan jumlah
    parameter yang dilatih.

    Args:
        model: model PyTorch mana pun.

    Returns:
        int jumlah elemen parameter.
    """
    return sum(p.numel() for p in model.parameters())


In [ ]:
def kd_loss_function(student_logits, teacher_logits, labels):
    """
    Menggabungkan loss distilasi terhadap teacher dengan loss terhadap label asli.

    Student belajar dari dua sumber sekaligus. Cross entropy menjaga student tetap
    terikat pada ground truth, sedangkan KL divergence memindahkan *dark knowledge*
    teacher: distribusi probabilitas penuh teacher menyimpan informasi tentang token
    mana yang "hampir benar", yang hilang bila hanya label keras yang dipakai.
    Bobot keduanya diatur `KD_ALPHA` (0.5 — porsi sama besar).

    `KD_TEMPERATURE` melunakkan kedua distribusi supaya probabilitas kecil ikut
    memberi sinyal. Pelunakan ini memperkecil gradien kira-kira sebesar `1/T**2`,
    karena itu hasil KL dikalikan `T**2` agar skalanya tetap sebanding dengan cross
    entropy dan `KD_ALPHA` benar-benar berperilaku sebagai bobot 50:50.

    Masking `labels != -100` membuang posisi padding sebelum KL dihitung. Tanpa ini
    ribuan posisi kosong ikut dirata-rata dan mengencerkan sinyal dari token asli —
    cross entropy sudah aman karena `ignore_index=-100`. Setelah masking, tensor
    menjadi berbentuk `[jumlah_token_valid, vocab]`, sehingga `reduction="batchmean"`
    membaginya per token dan satuannya setara dengan cross entropy.

    Args:
        student_logits: logits student, `[batch, seq_len, vocab]`.
        teacher_logits: logits teacher pada input yang sama.
        labels: target token; `-100` menandai posisi yang diabaikan.

    Returns:
        tensor skalar total loss.
    """
    vocab_size = student_logits.size(-1)

    ce_loss = F.cross_entropy(
        student_logits.view(-1, vocab_size), labels.view(-1), ignore_index=-100
    )

    mask = labels != -100
    student_valid = student_logits[mask]
    teacher_valid = teacher_logits[mask]

    kd_loss = F.kl_div(
        F.log_softmax(student_valid / KD_TEMPERATURE, dim=-1),
        F.softmax(teacher_valid / KD_TEMPERATURE, dim=-1),
        reduction="batchmean",
    ) * (KD_TEMPERATURE**2)

    total_loss = KD_ALPHA * kd_loss + (1 - KD_ALPHA) * ce_loss

    return total_loss


In [ ]:
def train_knowledge_distillation(
    student_model, teacher_model, train_loader, val_loader
):
    """
    Melatih student hasil pruning untuk memulihkan akurasi yang hilang.

    Pruning memotong neuron sekaligus memutus pola yang sudah dipelajari; tahap ini
    yang membuat sisa jaringan menyesuaikan diri. Teacher dibekukan
    (`eval()` + `requires_grad = False`) sehingga hanya berfungsi sebagai sumber
    logits acuan — pemanggilannya juga dibungkus `torch.no_grad()` agar graf
    komputasinya tidak ikut disimpan.

    Label tetap dioper ke kedua model bukan demi loss bawaan HuggingFace (yang memang
    diabaikan, hanya `.logits` yang dipakai), melainkan karena dari label itulah
    `decoder_input_ids` dibentuk lewat pergeseran ke kanan. Keduanya dengan demikian
    menerima teacher forcing yang identik, syarat agar logits student dan teacher
    memang sebanding posisi per posisi.

    Bobot terbaik dipilih berdasarkan validation loss, bukan epoch terakhir, agar
    model yang dievaluasi bukan model yang sudah overfit.

    Hyperparameter diambil dari config global: `KD_EPOCHS` (20), `KD_LR` (5e-5),
    `KD_TEMPERATURE` (2.0), dan `KD_ALPHA` (0.5).

    Args:
        student_model: model hasil pruning yang akan dilatih.
        teacher_model: model baseline sebagai acuan; tidak ikut berubah.
        train_loader: DataLoader split train.
        val_loader: DataLoader split validation untuk pemilihan bobot terbaik.

    Returns:
        tuple `(student_model, history)` — model dalam mode `eval()` dan daftar
        `{epoch, train_loss, val_loss}` per epoch untuk dicatat ke MLflow.
    """
    teacher_model.eval()
    for param in teacher_model.parameters():
        param.requires_grad = False

    student_model.train()

    teacher_model.to(TRAIN_DEVICE)
    student_model.to(TRAIN_DEVICE)

    optimizer = torch.optim.AdamW(student_model.parameters(), lr=KD_LR)

    best_val_loss = float("inf")
    best_state = None

    # KD Training
    history = []

    for epoch in range(KD_EPOCHS):
        student_model.train()
        total_train_loss = 0

        # Train KD LOSS
        for batch in train_loader:
            pixel_values = batch["pixel_values"].to(TRAIN_DEVICE)
            labels = batch["labels"].to(TRAIN_DEVICE)

            with torch.no_grad():
                teacher_outputs = teacher_model(
                    pixel_values=pixel_values, labels=labels, use_cache=False
                )

            student_outputs = student_model(
                pixel_values=pixel_values, labels=labels, use_cache=False
            )

            # Define the loss here
            loss = kd_loss_function(
                student_outputs.logits, teacher_outputs.logits, labels
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_loader)

        # Val KD LOSS
        student_model.eval()
        total_val_loss = 0

        with torch.no_grad():
            for batch in val_loader:
                pixel_values = batch["pixel_values"].to(TRAIN_DEVICE)
                labels = batch["labels"].to(TRAIN_DEVICE)

                teacher_outputs = teacher_model(
                    pixel_values=pixel_values, labels=labels, use_cache=False
                )

                student_outputs = student_model(
                    pixel_values=pixel_values, labels=labels, use_cache=False
                )

                val_loss = kd_loss_function(
                    student_outputs.logits, teacher_outputs.logits, labels
                )

                total_val_loss += val_loss.item()

            avg_val_loss = total_val_loss / len(val_loader)

            print(
                f"Epoch {epoch+1}/{KD_EPOCHS} - Train Loss: {avg_train_loss:.4f} - Val Loss: {avg_val_loss:.4f}"
            )

            if avg_val_loss < best_val_loss:
                best_val_loss = avg_val_loss
                best_state = copy.deepcopy(student_model.state_dict())

            history.append(
                {
                    "epoch": epoch + 1,
                    "train_loss": avg_train_loss,
                    "val_loss": avg_val_loss,
                }
            )

        if best_state is not None:
            student_model.load_state_dict(best_state)

    student_model.eval()

    return student_model, history


## 6 Dynamic Quantization

In [ ]:
def apply_dynamic_quantization_decoder(model):
    """
    Mengubah layer `nn.Linear` pada decoder menjadi INT8 dinamis.

    Model disalin lebih dulu agar versi FP32-nya tetap utuh — masih dibutuhkan sebagai
    sumber perhitungan FLOPs. Hanya decoder yang dikuantisasi, dan hasilnya wajib
    berjalan di CPU karena kernel INT8 dinamis tidak tersedia di CUDA.

    Args:
        model: model hasil KD yang akan dikuantisasi.

    Returns:
        salinan model dengan decoder terkuantisasi, berada di CPU dan mode `eval()`.
    """
    quantized_model = copy.deepcopy(model)
    quantized_model.to("cpu")
    quantized_model.eval()

    quantized_model.decoder = torch.quantization.quantize_dynamic(
        quantized_model.decoder, {nn.Linear}, dtype=torch.qint8
    )

    return quantized_model


In [ ]:
from torch.ao.nn.quantized.dynamic import Linear as QuantizedLinear


def check_quantization(model):
    """
    Menghitung jumlah layer decoder yang benar-benar berubah menjadi INT8.

    Verifikasi ini perlu karena `quantize_dynamic` tidak melempar error bila tidak ada
    layer yang cocok — hasil 0 berarti kuantisasi gagal diam-diam.

    Args:
        model: model hasil `apply_dynamic_quantization_decoder`.

    Returns:
        int jumlah `QuantizedLinear` pada decoder.
    """
    quantized_layers = 0

    for module in model.decoder.modules():
        if isinstance(module, QuantizedLinear):
            quantized_layers += 1

    print(f"Quantized Linear layers: {quantized_layers}")
    return quantized_layers


## 7 Experiment Tracking (MLflow & Databricks)

In [ ]:
def mlflow_metrics(metrics):
    """
    Menyaring dict metrik agar aman dikirim ke `mlflow.log_metrics`.

    MLflow hanya menerima float berhingga; nilai `None`, non-numerik, `NaN`, dan `inf`
    dibuang supaya satu metrik bermasalah tidak menggagalkan pencatatan seluruh run.

    Args:
        metrics: dict metrik mentah.

    Returns:
        dict berisi float berhingga saja.
    """
    cleaned = {}

    for key, value in metrics.items():
        if value is None:
            continue

        try:
            value = float(value)
        except (TypeError, ValueError):
            continue

        if np.isfinite(value):
            cleaned[key] = value

    return cleaned


In [ ]:
def mlflow_params(params):
    """
    Menyaring dict parameter agar aman dikirim ke `mlflow.log_params`.

    Parameter MLflow disimpan sebagai string, sehingga dict/list/tuple diserialisasi
    ke JSON dan nilai `None` dibuang.

    Args:
        params: dict parameter mentah.

    Returns:
        dict berisi nilai skalar atau string JSON.
    """
    cleaned = {}

    for key, value in params.items():
        if value is None:
            continue

        if isinstance(value, (dict, list, tuple)):
            value = json.dumps(value, ensure_ascii=False)

        cleaned[key] = value

    return cleaned


In [ ]:
def get_environment_information():
    """
    Mengumpulkan versi library dan spesifikasi perangkat saat eksperimen dijalankan.

    Disimpan sebagai artifact tiap run agar angka latensi punya konteks hardware dan
    hasil dapat ditelusuri ulang.

    Returns:
        dict informasi environment.
    """
    return {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "torch_version": torch.__version__,
        "transformers_version": transformers.__version__,
        "mlflow_version": mlflow.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_version": torch.version.cuda,
        "cudnn_version": torch.backends.cudnn.version(),
        "train_device": TRAIN_DEVICE,
        "eval_device": EVAL_DEVICE,
        "gpu_name": (
            torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
        ),
        "cpu_threads": torch.get_num_threads(),
        "seed": SEED,
    }


In [ ]:
def safe_model_name(model_name):
    """
    Mengubah nama model menjadi bentuk yang aman dipakai sebagai nama file/folder.

    Args:
        model_name: label model, mis. "DONUT-P50-KD".

    Returns:
        str tanpa karakter selain huruf, angka, `_`, `.`, dan `-`.
    """
    return re.sub(f"[^A-Za-z0-9_.-]+", "_", model_name).strip("_")


In [ ]:
def save_model_to_mlflow(model, model_name, is_quantized=False):
    """
    Menyimpan bobot, processor, dan metadata model sebagai artifact MLflow.

    Format penyimpanan dibedakan: model terkuantisasi disimpan utuh lewat `torch.save`
    (.pt) karena state_dict INT8 tidak bisa dimuat ulang ke arsitektur FP32, sedangkan
    model biasa disimpan sebagai safetensors lewat `save_pretrained`.

    `metadata.json` berisi `model_name` dan `is_quantized` — dua field inilah yang
    dibaca backend aplikasi untuk menentukan cara memuat model saat serving.

    Model dipindahkan ke CPU sebelum disimpan, lalu dikembalikan ke device asalnya
    kecuali bila terkuantisasi (yang memang harus tetap di CPU).

    Args:
        model: model yang disimpan.
        model_name: label model, dipakai sebagai nama folder artifact.
        is_quantized: menentukan format penyimpanan.

    Returns:
        None — artifact langsung dikirim ke run MLflow yang sedang aktif.
    """
    file_name = safe_model_name(model_name)

    try:
        original_device = next(model.parameters()).device

    except StopIteration:
        original_device = torch.device("cpu")

    model.to("cpu")
    model.eval()

    with tempfile.TemporaryDirectory() as temp_dir:
        artifact_root = Path(temp_dir) / file_name
        artifact_root.mkdir(parents=True, exist_ok=True)

        processor.save_pretrained(artifact_root / "processor")

        metadata = {
            "model_name": model_name,
            "base_model": MODEL_NAME,
            "framework": "pytorch",
            "is_quantized": is_quantized,
            "seed": SEED,
        }

        with open(artifact_root / "metadata.json", "w", encoding="utf-8") as file:

            json.dump(metadata, file, indent=2, ensure_ascii=False)
        if is_quantized:
            model.config.to_json_file(artifact_root / "config.json")

            torch.save(model, artifact_root / f"{file_name}.pt")

        else:
            model.save_pretrained(
                artifact_root / "huggingface_model", safe_serialization=True
            )

        mlflow.log_artifacts(str(artifact_root), artifact_path="model")

    if (not is_quantized) and (original_device.type != "cpu"):
        model.to(original_device)


In [ ]:
def add_baseline_comparison(result, baseline_result):
    """
    Menambahkan kolom selisih terhadap baseline pada satu baris hasil evaluasi.

    Konvensi tanda: kolom *Reduction* positif berarti lebih kecil/lebih cepat dari
    baseline, sedangkan `F1 Drop` dan `N-TED Increase` positif berarti kualitasnya
    menurun. FLOPs hanya dihitung bila nilai baseline tersedia dan bukan nol.

    Args:
        result: baris hasil dari `evaluate_model`.
        baseline_result: baris hasil model baseline.

    Returns:
        dict salinan `result` beserta kolom perbandingan.
    """
    compared_result = dict(result)

    compared_result["Size Reduction (%)"] = (
        (baseline_result["Size (MB)"] - result["Size (MB)"])
        / baseline_result["Size (MB)"]
        * 100
    )

    compared_result["Latency Reduction (%)"] = (
        (baseline_result["Latency (ms/sample)"] - result["Latency (ms/sample)"])
        / baseline_result["Latency (ms/sample)"]
        * 100
    )

    compared_result["F1 Drop"] = baseline_result["F1-Score"] - result["F1-Score"]

    compared_result["N-TED Increase"] = result["N-TED"] - baseline_result["N-TED"]

    baseline_flops = baseline_result.get("FLOPs (GFLOPs)")

    model_flops = result.get("FLOPs (GFLOPs)")

    if baseline_flops not in (None, 0) and model_flops is not None:
        compared_result["FLOPs Reduction (%)"] = (
            (baseline_flops - model_flops) / baseline_flops * 100
        )

    return compared_result


In [ ]:
def log_experiment_to_mlflow(
    model_name,
    model,
    result,
    technique,
    pruning_ratio=None,
    is_quantized=False,
    history=None,
    baseline_result=None,
    extra_params=None,
    extra_metrics=None,
):
    """
    Mencatat satu konfigurasi model sebagai satu run MLflow yang lengkap.

    Satu panggilan menghasilkan satu run berisi parameter, metrik, kurva loss KD per
    epoch, tag, artifact laporan (`result.json`, `environment.json`, `requirements.txt`),
    dan bobot modelnya. Semuanya dikumpulkan di satu tempat supaya tiap angka pada tabel
    hasil dapat ditelusuri ke run asalnya.

    Parameter yang tidak relevan diisi `None` dan otomatis tersaring: hyperparameter KD
    hanya tercatat bila `technique` mengandung "kd", `num_calibration_batches` hanya bila
    ada pruning. Kegagalan `pip freeze` ditangkap dan disimpan sebagai
    `requirements_error.txt` agar tidak membatalkan pencatatan run.

    Args:
        model_name: label model, dipakai sebagai `run_name`.
        model: model yang disimpan sebagai artifact.
        result: baris hasil dari `evaluate_model`.
        technique: deskripsi teknik kompresi; menentukan parameter mana yang dicatat.
        pruning_ratio: rasio pruning bila ada.
        is_quantized: menentukan format penyimpanan model.
        history: daftar loss per epoch dari `train_knowledge_distillation`.
        baseline_result: baris hasil baseline untuk menghitung kolom perbandingan.
        extra_params: parameter tambahan yang menimpa default.
        extra_metrics: metrik tambahan, mis. perbandingan khusus PTQ.

    Returns:
        dict `result` yang sudah dilengkapi `MLflow Run ID` dan `MLflow Artifact URI`.
    """

    if baseline_result is not None:
        result = add_baseline_comparison(result, baseline_result)
    else:
        result = dict(result)

    technique_lower = technique.lower()

    params = {
        "base_model": MODEL_NAME,
        "dataset": DATASET_NAME,
        "technique": technique,
        "seed": SEED,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "train_samples": len(train_data),
        "validation_samples": len(val_data),
        "test_samples": len(test_data),
        "evaluation_samples": EVAL_SAMPLES,
        "latency_samples": LATENCY_SAMPLES,
        "warmup_samples": WARMUP_SAMPLES,
        "train_device": TRAIN_DEVICE,
        "eval_device": EVAL_DEVICE,
        "cpu_threads": CPU_NUM_THREADS,
        "pruning_ratio": pruning_ratio,
        "num_calibration_batches": (
            NUM_CALIB_BATCHES if pruning_ratio is not None else None
        ),
        "kd_epochs": (KD_EPOCHS if "kd" in technique_lower else None),
        "kd_learning_rate": (KD_LR if "kd" in technique_lower else None),
        "kd_temperature": (KD_TEMPERATURE if "kd" in technique_lower else None),
        "kd_alpha": (KD_ALPHA if "kd" in technique_lower else None),
        "quantization": ("dynamic_int8_decoder_linear" if is_quantized else "none"),
    }

    if extra_params:
        params.update(extra_params)

    metric_mapping = {
        "Precision": "field_precision",
        "Recall": "field_recall",
        "F1-Score": "field_f1_score",
        "N-TED": "n_ted",
        "Size (MB)": "model_size_mb",
        "Latency (ms/sample)": "latency_ms_per_sample",
        "FLOPs (GFLOPs)": "estimated_gflops",
        "Size Reduction (%)": "size_reduction_pct",
        "Latency Reduction (%)": "latency_reduction_pct",
        "FLOPs Reduction (%)": "flops_reduction_pct",
        "F1 Drop": "f1_drop",
        "N-TED Increase": "n_ted_increase",
        "PTQ Size Reduction (%)": "ptq_size_reduction_pct",
        "PTQ Latency Reduction (%)": "ptq_latency_reduction_pct",
    }

    metrics = {
        mlflow_metric_name: result.get(result_name)
        for result_name, mlflow_metric_name in metric_mapping.items()
    }

    if extra_metrics:
        metrics.update(extra_metrics)

    with mlflow.start_run(run_name=model_name) as run:

        mlflow.log_metrics(mlflow_metrics(metrics))

        # Log train / validation loss KD per epoch
        if history:
            for history_row in history:

                mlflow.log_metrics(
                    mlflow_metrics(
                        {
                            "kd_train_loss": history_row.get("train_loss"),
                            "kd_validation_loss": history_row.get("val_loss"),
                        }
                    ),
                    step=history_row["epoch"],
                )

        mlflow.set_tags(
            {
                "experiment_group": EXPERIMENT_GROUP,
                "architecture": "DONUT",
                "framework": "PyTorch",
                "task": "Document Information Extraction",
                "dataset": "CORD-v2",
                "optimization": technique,
                "model_format": (
                    "pytorch_pt" if is_quantized else "huggingface_safetensors"
                ),
                "mlflow.note.content": (
                    "Evaluasi menggunakan official test split CORD-v2. "
                    "Latency diukur pada evaluation device yang dicatat "
                    "dalam parameter dan environment artifact."
                ),
            }
        )

        with tempfile.TemporaryDirectory() as temp_dir:
            report_directory = Path(temp_dir)

            with open(report_directory / "result.json", "w", encoding="utf-8") as file:

                json.dump(result, file, indent=2, ensure_ascii=False)

            with open(
                report_directory / "environment.json", "w", encoding="utf-8"
            ) as file:

                json.dump(
                    get_environment_information(), file, indent=2, ensure_ascii=False
                )

            try:
                package_versions = subprocess.check_output(
                    [sys.executable, "-m", "pip", "freeze"], text=True
                )

                (report_directory / "requirements.txt").write_text(
                    package_versions, encoding="utf-8"
                )

            except Exception as e:
                (report_directory / "requirements_error.txt").write_text(
                    str(e), encoding="utf-8"
                )

            mlflow.log_artifacts(str(report_directory), artifact_path="report")

        save_model_to_mlflow(
            model=model, model_name=model_name, is_quantized=is_quantized
        )

        run_id = run.info.run_id
        experiment_id = run.info.experiment_id
        artifact_uri = mlflow.get_artifact_uri()

    result["MLflow Run ID"] = run_id
    result["MLflow Artifact URI"] = artifact_uri

    print("\n Berhasil dicatat ke MLflow")
    print("Model  :", model_name)
    print("Run ID :", run_id)

    print(
        "Databricks URL:",
        f"{os.environ['DATABRICKS_HOST']}"
        f"/#mlflow/experiments/"
        f"{experiment_id}/runs/{run_id}",
    )

    return result

## 8. Eksekusi Eksperimen

In [ ]:
# Titik awal eksperimen: `results` menampung satu dict per konfigurasi model,
# dan seed di-reset supaya urutan sample serta inisialisasi dapat direproduksi.
results = []
reset_experiment_seed(SEED)


In [ ]:
# Baseline dijalankan paling awal karena hasilnya menjadi acuan pembanding
# (`baseline_result`) bagi seluruh model kompresi berikutnya.
baseline_result = evaluate_model("DONUT-Base", baseline_model)

baseline_result = log_experiment_to_mlflow(
    model_name="DONUT-Base",
    model=baseline_model,
    result=baseline_result,
    technique="baseline",
    pruning_ratio=None,
    is_quantized=False,
    history=None,
    baseline_result=None,
)

results.append(baseline_result)


In [ ]:
# Skor Taylor dihitung sekali dari baseline lalu dipakai ulang untuk ketiga rasio,
# sehingga perbedaan antar rasio murni berasal dari banyaknya neuron yang dibuang.
# `calibration_loader` sengaja tidak di-shuffle agar batch kalibrasinya selalu sama.
taylor_scores = collect_taylor_scores(
    baseline_model,
    calibration_loader,
    num_batches=NUM_CALIB_BATCHES,
    device=TRAIN_DEVICE,
)


In [ ]:
# folder gw berantakan jadi gw save link disini awkakwakwkawa

In [ ]:
# Loop utama eksperimen: tiap rasio pruning menghasilkan tiga run MLflow secara
# berurutan — DONUT-P{r} (pruning saja), DONUT-P{r}-KD (setelah distilasi), dan
# DONUT-P{r}-KD-Q (setelah kuantisasi). Tiap model dievaluasi dan dicatat sebelum
# menjadi input tahap berikutnya.
#
# `reset_experiment_seed` dipanggil sebelum tiap tahap stokastik (pruning dan KD)
# supaya hasil tiap rasio tidak bergantung pada rasio yang dijalankan sebelumnya.
# Di akhir iterasi model dihapus dan cache CUDA dikosongkan agar VRAM cukup untuk
# rasio berikutnya.
for ratio in PRUNING_RATIOS:
    ratio_name = int(ratio * 100)
    reset_experiment_seed(SEED)

    # Buat model hasil pruning
    pruned_model = build_pruned_model(
        baseline_model, taylor_scores, pruning_ratio=ratio
    )

    # Statistik parameter
    before_params = count_parameters(baseline_model)
    after_params = count_parameters(pruned_model)
    reduction = ((before_params - after_params) / before_params) * 100

    print(
        {
            "Target pruning": f"{ratio_name}%",
            "Before": before_params,
            "After": after_params,
            "Reduction": f"{reduction:.2f}%",
        }
    )

    # Evaluasi hasil pruning
    pruned_result = evaluate_model(f"DONUT-P{ratio_name}", pruned_model)

    pruned_result = log_experiment_to_mlflow(
        model_name=f"DONUT-P{ratio_name}",
        model=pruned_model,
        result=pruned_result,
        technique="taylor_pruning",
        pruning_ratio=ratio,
        is_quantized=False,
        baseline_result=baseline_result,
        extra_metrics={
            "parameters_before": before_params,
            "parameters_after": after_params,
            "parameter_reduction_pct": reduction,
        },
    )

    results.append(pruned_result)

    # KD Setelah pruning
    reset_experiment_seed(SEED)

    kd_model, kd_history = train_knowledge_distillation(
        pruned_model, baseline_model, train_loader, val_loader
    )
    kd_result = evaluate_model(f"DONUT-P{ratio_name}-KD", kd_model)

    kd_result = log_experiment_to_mlflow(
        model_name=f"DONUT-P{ratio_name}-KD",
        model=kd_model,
        result=kd_result,
        technique="taylor_pruning_kd",
        pruning_ratio=ratio,
        is_quantized=False,
        history=kd_history,
        baseline_result=baseline_result,
        extra_params={"source_pruning_run_id": pruned_result["MLflow Run ID"]},
        extra_metrics={
            "parameters_after_pruning": after_params,
            "parameter_reduction_pct": reduction,
        },
    )

    results.append(kd_result)

    # PTQ Setelah KD
    # NOTE: Dynamic INT8 quantization hanya punya kernel CPU (quantized::linear_dynamic
    # tidak tersedia di CUDA), jadi SEMUA pengukuran PTQ (before & after) dilakukan
    # di CPU agar perbandingan latensi apple-to-apple.
    before_quant_size = get_model_size_mb(kd_model)
    before_latency = measure_latency(
        kd_model, test_data, device="cpu", num_samples=LATENCY_SAMPLES
    )

    quantized_kd_model = apply_dynamic_quantization_decoder(kd_model)
    quantized_layer_count = check_quantization(quantized_kd_model)

    after_quant_size = get_model_size_mb(quantized_kd_model)
    after_latency = measure_latency(
        quantized_kd_model, test_data, device="cpu", num_samples=LATENCY_SAMPLES
    )

    quant_reduction = ((before_quant_size - after_quant_size) / before_quant_size) * 100
    latency_reduction = ((before_latency - after_latency) / before_latency) * 100

    print(
        {
            "PTQ Before MB": round(before_quant_size, 2),
            "PTQ After MB": round(after_quant_size, 2),
            "PTQ Reduction": f"{quant_reduction:.2f}%",
            "Latency Before": round(before_latency, 2),
            "Latency After": round(after_latency, 2),
            "Latency Reduction": f"{latency_reduction:.2f}%",
        }
    )

    quantized_kd_result = evaluate_model(
        f"DONUT-P{ratio_name}-KD-Q",
        quantized_kd_model,
        flops_source_model=kd_model,
        device="cpu",
    )
    quantized_kd_result["PTQ Size Reduction (%)"] = quant_reduction
    quantized_kd_result["PTQ Latency Reduction (%)"] = latency_reduction

    quantized_kd_result = log_experiment_to_mlflow(
        model_name=f"DONUT-P{ratio_name}-KD-Q",
        model=quantized_kd_model,
        result=quantized_kd_result,
        technique=("taylor_pruning_kd_" "dynamic_int8_quantization"),
        pruning_ratio=ratio,
        is_quantized=True,
        baseline_result=baseline_result,
        extra_params={
            "source_kd_run_id": kd_result["MLflow Run ID"],
            "quantized_module": "decoder",
            "quantized_layer_type": "torch.nn.Linear",
            "quantized_dtype": "torch.qint8",
        },
        extra_metrics={
            "quantized_linear_layers": quantized_layer_count,
            "ptq_size_before_mb": before_quant_size,
            "ptq_size_after_mb": after_quant_size,
            "ptq_latency_before_ms": before_latency,
            "ptq_latency_after_ms": after_latency,
        },
    )

    results.append(quantized_kd_result)

    del pruned_model
    del kd_model
    del quantized_kd_model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 9 Hasil & Analisis

In [ ]:
# Seluruh hasil disusun menjadi satu tabel perbandingan.
df_results = pd.DataFrame(results)
df_results


In [ ]:
# Nilai acuan diambil kembali dari baris baseline untuk menghitung ulang kolom
# perbandingan di level tabel.
baseline_size = df_results.loc[df_results["Model"] == "DONUT-Base", "Size (MB)"].values[
    0
]
baseline_latency = df_results.loc[
    df_results["Model"] == "DONUT-Base", "Latency (ms/sample)"
].values[0]
baseline_flops = df_results.loc[
    df_results["Model"] == "DONUT-Base", "FLOPs (GFLOPs)"
].values[0]
baseline_f1 = df_results.loc[df_results["Model"] == "DONUT-Base", "F1-Score"].values[0]
baseline_nted = df_results.loc[df_results["Model"] == "DONUT-Base", "N-TED"].values[0]


In [ ]:
# Kolom perbandingan dihitung ulang untuk seluruh baris, bukan hanya mengandalkan
# hasil `add_baseline_comparison` saat logging: baris baseline sendiri dicatat tanpa
# pembanding, sehingga di sini barulah kolomnya terisi 0.
df_results["Size Reduction (%)"] = (
    (baseline_size - df_results["Size (MB)"]) / baseline_size
) * 100

df_results["Latency Reduction (%)"] = (
    (baseline_latency - df_results["Latency (ms/sample)"]) / baseline_latency
) * 100

df_results["FLOPs Reduction (%)"] = (
    (baseline_flops - df_results["FLOPs (GFLOPs)"]) / baseline_flops
) * 100

df_results["F1 Drop"] = baseline_f1 - df_results["F1-Score"]
df_results["N-TED Increase"] = df_results["N-TED"] - baseline_nted

df_results


In [ ]:
# df_results.to_csv("hasil_eksperimen_donut_optimasi.csv", index=False)

In [ ]:
def log_final_experiment_table(df_results):
    """
    Mencatat tabel perbandingan seluruh model sebagai satu run ringkasan di MLflow.

    Run ini terpisah dari run per-model agar tabel akhir punya satu alamat tetap.
    Tabel disimpan dalam format CSV dan JSON; berkas JSON-nya yang kemudian dipakai
    aplikasi sebagai `hasil_eksperimen_donut_optimasi.json`.

    Args:
        df_results: DataFrame hasil seluruh konfigurasi model.

    Returns:
        None — artifact dan run dicatat langsung ke MLflow.
    """

    with mlflow.start_run(run_name="DONUT-ALL-MODELS-SUMMARY") as run:

        mlflow.set_tags(
            {
                "experiment_group": EXPERIMENT_GROUP,
                "run_type": "final_comparison",
                "architecture": "DONUT",
                "dataset": "CORD-v2",
            }
        )

        mlflow.log_params(
            {
                "total_model_configurations": len(df_results),
                "seed": SEED,
                "base_model": MODEL_NAME,
                "dataset": DATASET_NAME,
            }
        )

        with tempfile.TemporaryDirectory() as temp_dir:

            csv_path = Path(temp_dir) / "hasil_eksperimen_donut_optimasi.csv"

            json_path = Path(temp_dir) / "hasil_eksperimen_donut_optimasi.json"

            df_results.to_csv(csv_path, index=False)

            df_results.to_json(json_path, orient="records", indent=2)

            mlflow.log_artifact(str(csv_path), artifact_path="comparison")

            mlflow.log_artifact(str(json_path), artifact_path="comparison")

        print("Final comparison Run ID:", run.info.run_id)


log_final_experiment_table(df_results)
